# 07 — 전주 공식 라벨(08-25) 역추론 모델

**핵심 아이디어**: 공식 `train_labels.csv`의 08-25 라벨은 정답 구간이 **08-25 ~ 09-12**다.
본선 09-01의 정답 구간(**09-01 ~ 09-26**) 중 앞 12일(09-01 ~ 09-12)이 여기에 들어 있다.

- 08-25 정답 구간 중 **08-25 ~ 08-31 부분은 우리 데이터에 있다** (view_5d 공개).
- 그래서 "08-25 라벨 = 1인데 우리가 아는 앞부분 성과는 낮다" → **09-01 이후 영상이 잘 나왔다**는 뜻으로 역추론할 수 있다.

**학습 방법**: 같은 구조(7일 간격, 겹침 12일)를 과거 날짜로 재현해 학습한다.
기준일 D(07-26 ~ 08-05)마다 `prev = D−7일 시점 18일 창 라벨`(공식 규칙 재현)을 피처로 붙이고, 정답은 D 시점 26일 창 라벨.
예측 때는 `prev = 공식 08-25 라벨`, D = 09-01.

> ⚠ 08-25 공식 라벨은 09-01 이후 영상 성과로 계산된 값이다. 대회 규칙상 명시적 금지는 아니지만(배포 데이터),
> "기준일 이후 정보"에 해당하므로 사용 전 운영진 확인을 권장한다.

In [1]:
# ===== 라이브러리 · 경로 =====
import os, glob, platform, warnings
import duckdb
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score

warnings.filterwarnings('ignore')
RAW  = os.environ.get('EPOCH_RAW', '../data/raw/epoch_data')
SUB  = os.environ.get('EPOCH_SUB', '../submissions')
D    = '2026-09-01'     # 본선 기준일
PREV = '2026-08-25'     # 전주 공식 라벨 기준일 (D − 7일)
FUT, FUT_PREV = 26, 18  # 본선 / 공식 라벨 정답 구간 길이

# CSV를 파일 이름 그대로 DuckDB 뷰로 등록
con = duckdb.connect()
for p in sorted(glob.glob(f'{RAW}/*.csv')):
    con.sql(f"CREATE OR REPLACE VIEW {os.path.splitext(os.path.basename(p))[0]} AS "
            f"SELECT * FROM read_csv('{p}', header=true, auto_detect=true)")

## 1. 기본 피처 (05번과 동일한 23개, `published_at < 기준일`만 사용)

In [2]:
def make_features(Dref):
    """기준일 Dref 이전 영상만으로 채널별 피처 (05번 노트북과 동일)"""
    f = con.sql(f"""
    WITH lv AS (   -- 기준일 이전 롱폼 + 며칠 전 영상인지
        SELECT channel_id, view_5d, like_5d, comment_5d,
               DATE_DIFF('hour', published_at, TIMESTAMP '{Dref}') / 24.0 AS days_ago
        FROM video_5d_views
        WHERE is_shorts = 0 AND view_5d IS NOT NULL AND published_at < TIMESTAMP '{Dref}'),
    v5 AS (        -- 롱폼 성과 요약
        SELECT channel_id, COUNT(*) AS n_long_5d, MEDIAN(view_5d) AS past_med,
               MEDIAN(view_5d) FILTER (WHERE days_ago < 7)   AS med_7,
               MEDIAN(view_5d) FILTER (WHERE days_ago < 14)  AS med_14,
               MEDIAN(view_5d) FILTER (WHERE days_ago >= 14) AS med_old,
               MEDIAN(like_5d / NULLIF(view_5d, 0))          AS like_rate,
               MEDIAN(comment_5d / NULLIF(view_5d, 0))       AS comment_rate,
               STDDEV(LN(1 + view_5d))                       AS log_view_std,
               QUANTILE_CONT(LN(1 + view_5d), 0.1)           AS log_q10,
               QUANTILE_CONT(LN(1 + view_5d), 0.9)           AS log_q90,
               REGR_SLOPE(LN(1 + view_5d), -days_ago)        AS trend_slope,
               MIN(days_ago)                                 AS days_since_meas
        FROM lv GROUP BY 1),
    sv AS (        -- 쇼츠 성과
        SELECT channel_id, MEDIAN(view_5d) AS short_med FROM video_5d_views
        WHERE is_shorts = 1 AND view_5d IS NOT NULL AND published_at < TIMESTAMP '{Dref}' GROUP BY 1),
    up AS (        -- 업로드 활동
        SELECT channel_id,
               COUNT(*) FILTER (WHERE is_shorts = 0 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 7 DAY)  AS n_long_7,
               COUNT(*) FILTER (WHERE is_shorts = 0 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 14 DAY) AS n_long_14,
               COUNT(*) FILTER (WHERE is_shorts = 0 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 28 DAY) AS n_long_28,
               COUNT(*) FILTER (WHERE is_shorts = 1 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 14 DAY) AS n_short_14,
               AVG(duration_sec) FILTER (WHERE is_shorts = 0) AS avg_dur,
               DATE_DIFF('hour', MAX(published_at) FILTER (WHERE is_shorts = 0), TIMESTAMP '{Dref}') / 24.0 AS days_since_long
        FROM videos WHERE published_at < TIMESTAMP '{Dref}' GROUP BY 1),
    games AS (     -- Steam 게임 매핑 수
        SELECT v.channel_id, COUNT(DISTINCT vg.app_id) AS n_games
        FROM videos v LEFT JOIN video_games vg USING (video_id)
        WHERE v.published_at < TIMESTAMP '{Dref}' GROUP BY 1)
    SELECT * FROM v5 LEFT JOIN sv USING (channel_id) LEFT JOIN up USING (channel_id) LEFT JOIN games USING (channel_id)
    """).df()
    lp = np.log1p(f.past_med)
    f['log_past_med']      = lp
    f['mom_7']             = np.log1p(f.med_7) - lp
    f['mom_14']            = np.log1p(f.med_14) - lp
    f['mom_old']           = np.log1p(f.med_14) - np.log1p(f.med_old)
    f['short_ratio']       = f.n_short_14 / (f.n_short_14 + f.n_long_14).replace(0, np.nan)
    f['log_short_med_rel'] = np.log1p(f.short_med) - lp
    f['has_game']          = (f.n_games > 0).astype(int)
    f['log_n_long_5d']     = np.log1p(f.n_long_5d)
    return f

F23 = ['log_past_med', 'n_long_5d', 'like_rate', 'n_long_14', 'n_short_14', 'short_ratio', 'days_since_long',
       'mom_14', 'log_view_std', 'n_games', 'has_game', 'mom_7', 'mom_old', 'comment_rate',
       'log_q10', 'log_q90', 'trend_slope', 'days_since_meas', 'n_long_7', 'n_long_28', 'avg_dur',
       'log_short_med_rel', 'log_n_long_5d']

## 2. 라벨 재현 + 전주 라벨 역추론 피처

| 피처 | 의미 |
|---|---|
| `prev_f` | 전주(D−7) 라벨 (없으면 기저율 0.2) |
| `has_prev` | 전주 표본에 있었는지 |
| `log_prev_past` | 전주 기준 past 중앙값 |
| `n_known`, `known_rel_prev` | 전주 정답 구간 중 **이미 아는 부분** [D−7, D)의 롱폼 수 · 성장 |
| `known_rel_now` | 그 부분이 현재 past 대비 어떤지 |
| `prev_x_known` | 라벨=1인데 아는 부분이 낮을수록 ↑ → D 이후 영상이 잘 나왔다는 신호 |
| `prev_x_lowpast` | 전주 라벨 × 낮은 과거 수준 |

In [3]:
def make_labels(Dref, fut):
    """공식 라벨 규칙 재현: 롱폼, past 3편 이상 · past_score ≥ 100, 기준일 내 growth 상위 20% → 1"""
    return con.sql(f"""
        WITH agg AS (
          SELECT channel_id,
                 MEDIAN(view_5d) FILTER (WHERE published_at < TIMESTAMP '{Dref}') AS past_score,
                 COUNT(view_5d)  FILTER (WHERE published_at < TIMESTAMP '{Dref}') AS n_past,
                 COALESCE(MEDIAN(view_5d) FILTER (WHERE published_at >= TIMESTAMP '{Dref}'
                       AND published_at < TIMESTAMP '{Dref}' + INTERVAL {fut} DAY), 0) AS future_score
          FROM video_5d_views WHERE is_shorts = 0 AND view_5d IS NOT NULL GROUP BY 1),
        scored AS (SELECT channel_id, LN(1 + future_score) - LN(1 + past_score) AS growth
                   FROM agg WHERE n_past >= 3 AND past_score >= 100)
        SELECT channel_id, CASE WHEN growth >= QUANTILE_CONT(growth, 0.8) OVER () THEN 1 ELSE 0 END AS target
        FROM scored""").df()


def add_prev_features(df, Dref, prev_labels):
    """전주 라벨 + 전주 정답 구간 중 이미 아는 부분 [D−7, D)의 성과를 붙인다"""
    P = (pd.Timestamp(Dref) - pd.Timedelta(days=7)).strftime('%Y-%m-%d')
    pl = prev_labels[['channel_id', 'target']].rename(columns={'target': 'prev'})
    # 전주 기준 past 중앙값
    pp = con.sql(f"""SELECT channel_id, MEDIAN(view_5d) AS prev_past FROM video_5d_views
                     WHERE is_shorts = 0 AND view_5d IS NOT NULL AND published_at < TIMESTAMP '{P}' GROUP BY 1""").df()
    # 이미 아는 부분: [P, Dref) 업로드 롱폼
    kn = con.sql(f"""SELECT channel_id, COUNT(*) AS n_known, MEDIAN(view_5d) AS known_med FROM video_5d_views
                     WHERE is_shorts = 0 AND view_5d IS NOT NULL
                       AND published_at >= TIMESTAMP '{P}' AND published_at < TIMESTAMP '{Dref}' GROUP BY 1""").df()
    df = df.merge(pl, on='channel_id', how='left').merge(pp, on='channel_id', how='left').merge(kn, on='channel_id', how='left')
    df['n_known']        = df.n_known.fillna(0)
    df['has_prev']       = df.prev.notna().astype(int)
    df['prev_f']         = df.prev.fillna(0.2)
    df['log_prev_past']  = np.log1p(df.prev_past)
    df['known_rel_prev'] = np.log1p(df.known_med) - df.log_prev_past
    df['known_rel_now']  = np.log1p(df.known_med) - df.log_past_med
    df['prev_x_known']   = df.prev_f * (1 - df.known_rel_prev.fillna(0).clip(-3, 3) / 3)
    df['prev_x_lowpast'] = df.prev_f * (-df.log_past_med)
    return df

PAIR = F23 + ['prev_f', 'has_prev', 'log_prev_past', 'n_known', 'known_rel_prev', 'known_rel_now',
              'prev_x_known', 'prev_x_lowpast']


def build_train(Dref):
    """학습 1개 기준일: D 시점 26일 라벨 + D 피처 + (D−7) 18일 재현 라벨 기반 역추론 피처"""
    P = (pd.Timestamp(Dref) - pd.Timedelta(days=7)).strftime('%Y-%m-%d')
    df = make_labels(Dref, FUT).merge(make_features(Dref), on='channel_id', how='left')
    df = add_prev_features(df, Dref, make_labels(P, FUT_PREV))
    df['ref_date'] = Dref
    return df

# 학습 기준일: D+26일이 데이터 끝(08-31) 안에 들어오는 07-26 ~ 08-05
DATES = pd.date_range('2026-07-26', '2026-08-05', freq='D').strftime('%Y-%m-%d').tolist()
train = pd.concat([build_train(d) for d in DATES], ignore_index=True)
print('학습 행 수:', len(train), '| 양성 비율:', round(train.target.mean(), 3))

학습 행 수: 5102 | 양성 비율: 0.201


## 3. 모델 · 검증

- `HistGradientBoosting`(깊이 3) 5개 시드 평균 + 로지스틱 회귀, **순위 평균**
- 검증: 테스트 기준일보다 4일 이상 앞선 기준일로만 학습 → 08-01 ~ 08-05 각각 평가

In [4]:
def hgb(seed):
    return HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=300, min_samples_leaf=30,
                                          l2_regularization=1.0, random_state=seed)

def logreg():
    return Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler()),
                     ('m', LogisticRegression(max_iter=3000, C=0.1, class_weight='balanced'))])

def rank01(x):
    x = pd.Series(np.asarray(x, float))
    return rankdata(x.fillna(x.median())) / len(x)

def predict(tr, te):
    """HGB 5시드 평균 순위 + 로지스틱 순위의 평균"""
    p_hgb = np.mean([hgb(s).fit(tr[PAIR], tr.target).predict_proba(te[PAIR])[:, 1] for s in range(5)], axis=0)
    p_lr  = logreg().fit(tr[PAIR], tr.target).predict_proba(te[PAIR])[:, 1]
    return (rank01(p_hgb) + rank01(p_lr)) / 2, p_hgb

rows = []
for td in ['2026-08-01', '2026-08-02', '2026-08-03', '2026-08-04', '2026-08-05']:
    te = train[train.ref_date == td]
    tr = train[train.ref_date <= (pd.Timestamp(td) - pd.Timedelta(days=4)).strftime('%Y-%m-%d')]
    s, _ = predict(tr, te)
    base = logreg().fit(tr[F23], tr.target).predict_proba(te[F23])[:, 1]     # 05번 방식(피처 23개) 비교
    rows.append({'평가 기준일': td, '학습 행': len(tr),
                 '05방식 PR-AUC': round(average_precision_score(te.target, base), 3),
                 '전주라벨만': round(average_precision_score(te.target, te.prev_f + 1e-3 * rank01(-te.log_past_med)), 3),
                 '07 모델 PR-AUC': round(average_precision_score(te.target, s), 3)})
val = pd.DataFrame(rows)
val.loc['평균'] = ['', '', val['05방식 PR-AUC'].mean().round(3), val['전주라벨만'].mean().round(3), val['07 모델 PR-AUC'].mean().round(3)]
val

,평가 기준일,학습 행,05방식 PR-AUC,전주라벨만,07 모델 PR-AUC
0,2026-08-01,1152,0.379,0.429,0.546
1,2026-08-02,1582,0.375,0.481,0.552
2,2026-08-03,2030,0.388,0.424,0.538
3,2026-08-04,2509,0.416,0.473,0.580
4,2026-08-05,3003,0.422,0.445,0.580
평균,,,0.396,0.450,0.559


## 4. 09-01 예측 (전주 라벨 = 공식 08-25)

In [5]:
# 공식 08-25 라벨
tl = pd.read_csv(f'{RAW}/train_labels.csv')
tl['channel_id'] = tl.row_id.str.rsplit('_', n=1).str[0]
prev_off = tl[tl.row_id.str.endswith(PREV)][['channel_id', 'target']]

# 채점 대상 694행 + 09-01 피처 + 역추론 피처
sample = pd.read_csv(f'{RAW}/sample_submission.csv', encoding='utf-8-sig')
sample['channel_id'] = sample.row_id.str.rsplit('_', n=1).str[0]
test = sample[['row_id', 'channel_id']].merge(make_features(D), on='channel_id', how='left')
test = add_prev_features(test, D, prev_off)
print('전주 라벨 있는 채널:', test.has_prev.sum(), '/', len(test), '| 아는 부분 영상 있는 채널:', (test.n_known > 0).sum())

# 전체 학습 기준일로 학습 → 예측
score, p_hgb = predict(train, test)
score = score + 1e-6 * rank01(p_hgb)                       # 동점 깨기
test['prediction'] = (score - score.min()) / (score.max() - score.min())
sub = sample[['row_id']].merge(test[['row_id', 'prediction']], on='row_id', how='left')

# 제출 가이드 §3 검사
assert list(sub.columns) == ['row_id', 'prediction'] and len(sub) == 694
assert set(sub.row_id) == set(sample.row_id) and sub.row_id.is_unique
assert sub.prediction.notna().all() and sub.prediction.between(0, 1).all()

FILE = f'{SUB}/submit_07_prevlabel_inversion_hgb.csv'
sub.to_csv(FILE, index=False, encoding='utf-8')
print('저장:', FILE)
sub.describe()

전주 라벨 있는 채널: 666 / 694 | 아는 부분 영상 있는 채널: 598


저장: ../submissions/submit_07_prevlabel_inversion_hgb.csv


,prediction
count,694.000000
mean,0.497080
std,0.273947
min,0.000000
25%,0.266423
50%,0.476277
75%,0.747263
max,1.000000


## 5. 정리

- 05번(피처만) 대비 과거 날짜 재현 검증에서 PR-AUC 평균 **약 +0.16** 상승 (5개 기준일 모두 일관).
- 기대 리더보드: 05번 0.4554 기준 **0.58 ~ 0.62** 추정 (리더보드 ±0.03 흔들림 별도).
- 상한: 09-01 정답 구간의 앞 12일을 **완벽히** 알아도 PR-AUC ≈ 0.65 ~ 0.70 (재현 실험). 전주 라벨은 0/1 한 비트라 그보다 낮다.